#Initializations

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import trim, col

#Read Bronze Layer

In [0]:
df = spark.table("workspace.bronze.erp_loc_a101")

#Silver Transformations

##Trimming leading & trailing spaces

In [0]:
#Instead of reconstructing the DataFrame plan for each column using a for loop, this method updates all string columns in a single plan, making it faster.
#.alias(file.name) - keeps the origianl name of the column instead of "trim(prd_id)" for example

# Old Code:
# for field in df.schema.fields:
#     if isinstance(field.dataType, StringType):
#         df = df.withColumn(field.name, trim(col(field.name)))


df = df.select([
    trim(col(field.name)).alias(field.name) if isinstance(field.dataType, StringType) 
    else col(field.name) 
    for field in df.schema.fields
])

#Customer ID Cleanup

In [0]:
df = df.withColumn("cid", F.regexp_replace(col("cid"), "-", ""))

#Country Normalization

In [0]:
df = df.withColumn(
    "cntry",
    F.when(col("cntry") == "DE", "Germany")
     .when(col("cntry").isin("US", "USA"), "United States")
     .when((col("cntry") == "") | col("cntry").isNull(), "n/a")
     .otherwise(col("cntry"))
)

#Renaming Columns

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "cntry": "country"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

#Sanity checks of dataframe

In [0]:
df.limit(10).display()

#Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customer_location")

#Sanity checks of the Silver table

##Data Standardization & Consistency

In [0]:
%sql
SELECT DISTINCT
country FROM silver.erp_customer_location ORDER BY country

##Check Both Customer number & Country Normalization

In [0]:
%sql
SELECT * FROM workspace.silver.erp_customer_location LIMIT 30